# Debugging & Optimizing RPA for Flight Booking Automation

### A portfolio case study in reliable Python automation

This notebook walks through a realistic automation-engineering lifecycle:

**Problem statement → broken workflow → debugging → optimized design → validation → measured results**

The scenario is intentionally simulated. No live airline reservation system, passenger database, or payment service is used.

> **Portfolio objective:** demonstrate how a brittle automation can be diagnosed and redesigned using validation, exception handling, retry logic, logging, testing, reproducibility, and performance measurement.

## 1. Business Problem

A travel operations team receives reservation records that must be validated and sent through a booking-confirmation workflow.

A fragile bot can fail when it encounters:

- missing or malformed booking fields,
- non-numeric fares,
- unsupported airport codes,
- duplicate PNRs,
- invalid routes,
- booking statuses that should not be processed,
- temporary downstream service failures.

A reliable automation should **fail safely, explain what happened, retry only transient failures, and expose measurable run outcomes**.

## 2. Engineering Success Criteria

The improved workflow should:

1. Validate inputs before downstream processing.
2. Normalize common input formats such as `"$250.50"`.
3. Reject invalid records without crashing the entire batch.
4. Suppress duplicate PNRs.
5. Skip non-processable statuses according to explicit business rules.
6. Retry simulated transient confirmation failures with bounded attempts.
7. Produce logs and summary metrics.
8. Be reproducible and testable.
9. Support measured—not invented—performance comparisons.

In [ ]:
from pathlib import Path
import sys
import random
import time
from statistics import mean

# Support execution from either the repository root or a notebooks-style location.
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.flight_booking_rpa import (
    ValidationError,
    deduplicate,
    generate_reservations,
    run_pipeline,
    validate_reservation,
)

print(f"Project root: {ROOT}")

## 3. Reproducible Synthetic Reservations

The original notebook generated random data without a fixed seed. That makes debugging harder because the dataset changes from run to run.

The improved generator accepts a seed, allowing the same test case to be reproduced.

In [ ]:
reservations = generate_reservations(10, seed=42)
reservations[:3]

In [ ]:
# Reproducibility check
assert generate_reservations(5, seed=42) == generate_reservations(5, seed=42)
print("PASS: seeded reservation generation is reproducible.")

## 4. Intentionally Broken Automation

The following function represents the kind of brittle workflow we want to improve.

It assumes every field exists, converts fares directly, performs no airport validation, does not detect duplicate PNRs, and has no per-record error isolation.

**This cell defines the broken implementation; it does not execute it yet.**

In [ ]:
def broken_process_reservations(records):
    processed = []

    for record in records:
        # Fragile assumptions:
        fare = float(record["Fare"])
        route = f'{record["Origin"]}-{record["Destination"]}'

        # No PNR, airport, duplicate, or route validation.
        if record["Status"] == "Confirmed":
            processed.append({
                "PNR": record["PNR"],
                "Route": route,
                "Fare": fare,
            })

    return processed

### Controlled bad data

We now create specific failure cases rather than relying on random chance. This makes the debugging story reproducible.

In [ ]:
bad_records = [
    {
        "PNR": "ABC123", "Passenger": "Valid Passenger",
        "Origin": "JFK", "Destination": "LAX",
        "Fare": "245.50", "Status": "Confirmed",
    },
    {
        "PNR": "BAD12", "Passenger": "Bad PNR",
        "Origin": "JFK", "Destination": "LAX",
        "Fare": 200, "Status": "Confirmed",
    },
    {
        "PNR": "DEF456", "Passenger": "Bad Fare",
        "Origin": "LHR", "Destination": "DXB",
        "Fare": "not-a-number", "Status": "Confirmed",
    },
    {
        "PNR": "GHI789", "Passenger": "Bad Route",
        "Origin": "JFK", "Destination": "JFK",
        "Fare": 190, "Status": "Confirmed",
    },
    {
        "PNR": "JKL012", "Passenger": "Bad Airport",
        "Origin": "XXX", "Destination": "DXB",
        "Fare": 230, "Status": "Confirmed",
    },
]

print(f"Controlled test records: {len(bad_records)}")

In [ ]:
try:
    broken_process_reservations(bad_records)
except Exception as exc:
    print(f"Broken workflow stopped with {type(exc).__name__}: {exc}")

## 5. Debugging the Failure Modes

Instead of allowing one malformed row to terminate the batch, validate each reservation independently and capture the reason for rejection.

In [ ]:
validation_results = []

for record in bad_records:
    try:
        normalized = validate_reservation(record)
        validation_results.append({
            "PNR": record.get("PNR"),
            "Result": "VALID",
            "Reason": "Passed validation",
        })
    except ValidationError as exc:
        validation_results.append({
            "PNR": record.get("PNR"),
            "Result": "REJECTED",
            "Reason": str(exc),
        })

for result in validation_results:
    print(result)

### What changed?

The validation layer now checks:

- required fields,
- six-character alphanumeric PNR format,
- supported airport codes,
- origin/destination conflicts,
- positive numeric fares,
- supported booking statuses.

This separates **data-quality failures** from **transient operational failures**. Invalid input should fail fast; temporary service problems may be retried.

## 6. Duplicate Handling

A booking automation should avoid processing the same PNR more than once in a batch.

In [ ]:
duplicate_input = [
    {
        "PNR": "ABC123", "Passenger": "Passenger One",
        "Origin": "JFK", "Destination": "LAX",
        "Fare": 200, "Status": "Confirmed",
    },
    {
        "PNR": "ABC123", "Passenger": "Duplicate Copy",
        "Origin": "JFK", "Destination": "LAX",
        "Fare": 200, "Status": "Confirmed",
    },
]

validated = [validate_reservation(record) for record in duplicate_input]
unique = deduplicate(validated)

print(f"Before deduplication: {len(validated)}")
print(f"After deduplication:  {len(unique)}")

## 7. Optimized Automation Pipeline

The reusable implementation in `src/flight_booking_rpa.py` orchestrates validation, deduplication, status decisions, simulated confirmation calls, bounded retries, logging, and metrics.

For this demonstration, the transient failure rate is deliberately elevated so retry behavior is easier to observe. This is a **simulation parameter**, not a real airline failure rate.

In [ ]:
demo_records = generate_reservations(50, seed=42)

summary = run_pipeline(
    demo_records,
    seed=42,
    transient_failure_rate=0.20,
    max_attempts=3,
    base_delay_seconds=0.0,
)

summary

## 8. Validation Checks

These assertions act as lightweight notebook-level acceptance tests. The repository also contains a formal unit-test suite in `tests/`.

In [ ]:
clean_record = {
    "PNR": "ZXCV12",
    "Passenger": "Test Passenger",
    "Origin": "JFK",
    "Destination": "LAX",
    "Fare": "$250.50",
    "Status": "Confirmed",
}

normalized = validate_reservation(clean_record)

assert normalized.pnr == "ZXCV12"
assert normalized.fare == 250.50
assert normalized.origin != normalized.destination

metrics = run_pipeline(
    [clean_record],
    seed=1,
    transient_failure_rate=0.0,
    base_delay_seconds=0.0,
)

assert metrics["input_records"] == 1
assert metrics["valid_records"] == 1
assert metrics["processed"] == 1
assert metrics["failed"] == 0

print("PASS: notebook acceptance checks completed successfully.")

## 9. Measured Performance Comparison

This section compares two duplicate-detection approaches on the **same deterministic 10,000-record dataset**.

- **Naive approach:** tracks seen PNRs in a Python list, making membership checks progressively more expensive.
- **Optimized approach:** uses the project's set-based deduplication.

The result is measured when you run the notebook. No performance percentage is hard-coded.

In [ ]:
def naive_validation(records):
    output = []
    seen = []

    for record in records:
        try:
            validated = validate_reservation(record)
        except ValidationError:
            continue

        if validated.pnr not in seen:
            seen.append(validated.pnr)
            output.append(validated)

    return output


def optimized_validation(records):
    valid = []

    for record in records:
        try:
            valid.append(validate_reservation(record))
        except ValidationError:
            continue

    return deduplicate(valid)


def measure(function, records, repeats=5):
    durations = []

    for _ in range(repeats):
        start = time.perf_counter()
        function(records)
        durations.append(time.perf_counter() - start)

    return mean(durations)


benchmark_records = generate_reservations(10_000, seed=42)

naive_seconds = measure(naive_validation, benchmark_records)
optimized_seconds = measure(optimized_validation, benchmark_records)

runtime_change = (
    (naive_seconds - optimized_seconds) / naive_seconds * 100
    if naive_seconds
    else 0.0
)

print(f"Records: {len(benchmark_records):,}")
print(f"Naive mean runtime:     {naive_seconds:.6f} seconds")
print(f"Optimized mean runtime: {optimized_seconds:.6f} seconds")
print(f"Measured runtime change: {runtime_change:.2f}%")

### Interpreting the benchmark

The benchmark isolates one engineering decision: list-based versus set-based duplicate tracking. It is **not** a claim about total end-to-end airline booking latency.

Results vary by machine, operating system, Python version, and current system load. The important portfolio practice is that the measurement is reproducible and the comparison methodology is visible.

## 10. Final Architecture

```text
Synthetic / CSV Reservation Input
              │
              ▼
     Required Field Checks
              │
              ▼
   Normalize Fare & Codes
              │
              ▼
     Business Rule Checks
              │
              ▼
       Deduplicate PNR
              │
              ▼
     Processable Status?
        │           │
       No          Yes
        │           │
        ▼           ▼
   Skip + Log   Confirmation Call
                    │
                    ▼
              Transient Failure?
                 │       │
                Yes      No
                 │       │
                 ▼       ▼
             Retry     Success
                 │       │
                 └───┬───┘
                     ▼
              Summary Metrics
```

## 11. Engineering Takeaways

This project demonstrates that debugging an automation is more than correcting a single exception.

A resilient workflow needs:

- **input contracts** so malformed data is identified early,
- **failure classification** so permanent errors are not retried,
- **bounded retries** for transient service failures,
- **idempotency-oriented duplicate handling**,
- **observability** through logs and metrics,
- **reproducibility** through deterministic test data,
- **automated tests and CI**,
- **measured performance evidence** rather than unsupported optimization claims.

These practices make the automation easier to operate, troubleshoot, test, and extend.

## 12. Repository Components

| Component | Purpose |
|---|---|
| `src/flight_booking_rpa.py` | Reusable automation implementation |
| `tests/test_flight_booking_rpa.py` | Unit tests |
| `data/sample_reservations.csv` | Synthetic mixed-quality test data |
| `scripts/benchmark.py` | Standalone reproducible benchmark |
| `docs/architecture.md` | Architecture and design decisions |
| `.github/workflows/tests.yml` | Multi-version Python CI |
| This notebook | End-to-end portfolio walkthrough |

### Run the formal test suite

```bash
python -m unittest discover -s tests -v
```

### Run the standalone benchmark

```bash
python scripts/benchmark.py
```

---

**Project scope:** educational and portfolio simulation. No live airline systems or real passenger records are used.